# Lezione 7F — Soluzioni: Agno

**Corso**: Programmazione di Applicazioni Intelligenti  
**Tipo**: Soluzioni dell'esercitazione 7E


In [ ]:
# === Setup ===

!pip install -q agno[mistral,sqlite] ddgs wikipedia

import os
from google.colab import userdata
os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")

print("Setup completato!")


---
## Soluzione Esercizio 1 — Primo agente Agno

In [ ]:
from agno.agent import Agent
from agno.models.mistral import MistralChat

def calculate(expression: str) -> str:
    """Calcola un'espressione matematica. Supporta +, -, *, / e parentesi."""
    allowed = set("0123456789+-*/.(). ")
    if not all(c in allowed for c in expression):
        return "Errore: espressione non valida"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Errore: {e}"

agent = Agent(
    model=MistralChat(id="mistral-small-latest"),
    tools=[calculate],
    instructions=["Sei un assistente matematico. Usa il tool calculate per i calcoli."],
    markdown=True,
)

agent.print_response("Quanto fa 847 * 23 + 156?", stream=True)


---
## Soluzione Esercizio 2 — Progetta un assistente con tool e memoria

> **Nota**: questa è *una* possibile soluzione. La tua potrebbe avere tool diversi ed essere ugualmente valida. L'importante è che i tool siano coerenti con lo scenario e la memoria funzioni.

In [ ]:
# === Parte A: una possibile implementazione per lo scenario "tutor universitario" ===

from agno.agent import Agent
from agno.models.mistral import MistralChat
from agno.db.sqlite import SqliteDb
from datetime import datetime, date

# Tool 1: calcola i giorni che mancano a una data
def days_until(target_date: str) -> str:
    """Calcola quanti giorni mancano a una data. Formato: YYYY-MM-DD (es. '2026-07-15')."""
    try:
        target = date.fromisoformat(target_date)
        delta = (target - date.today()).days
        if delta < 0:
            return f"La data {target_date} e' gia' passata ({abs(delta)} giorni fa)"
        return f"Mancano {delta} giorni al {target_date}"
    except ValueError:
        return "Errore: usa il formato YYYY-MM-DD"

# Tool 2: calcolatrice (per crediti, medie, ecc.)
def calculate(expression: str) -> str:
    """Calcola un'espressione matematica. Supporta +, -, *, / e parentesi."""
    allowed = set("0123456789+-*/.(). ")
    if not all(c in allowed for c in expression):
        return "Errore: espressione non valida"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Errore: {e}"

# Tool 3 (bonus): suggerisce come dividere il tempo di studio
def study_planner(hours_available: float, num_topics: int) -> str:
    """Suggerisce come dividere il tempo di studio tra i vari argomenti.
    hours_available: ore totali disponibili, num_topics: numero di argomenti da studiare."""
    if num_topics <= 0:
        return "Errore: serve almeno 1 argomento"
    mins_per_topic = int((hours_available * 60) / num_topics)
    pause = 5 if mins_per_topic > 30 else 0
    return (
        f"Con {hours_available}h e {num_topics} argomenti: "
        f"~{mins_per_topic} min per argomento"
        + (f" (inclusi {pause} min di pausa)" if pause else "")
    )

# L'agente tutor
tutor = Agent(
    model=MistralChat(id="mistral-small-latest"),
    tools=[days_until, calculate, study_planner],
    instructions=[
        "Sei un tutor universitario amichevole e organizzato.",
        "Aiuti lo studente a pianificare lo studio per gli esami.",
        "Usa days_until per calcolare i giorni mancanti agli esami.",
        "Usa calculate per calcoli su crediti, medie, o durate.",
        "Usa study_planner per suggerire come dividere il tempo.",
        "Rispondi in italiano, in modo chiaro e motivante.",
    ],
    db=SqliteDb(db_file="/tmp/agno_tutor.db"),
    add_history_to_context=True,
    num_history_runs=5,
    markdown=True,
)

print("Tutor creato! Esegui la cella successiva per la conversazione.")


In [ ]:
# === Parte B: Loop interattivo ===
# Conversazione di esempio:
# 1. "Il mio esame di AI e' il 2026-07-10, quanto manca?"
# 2. "Ho 5 argomenti da studiare e 3 ore oggi, come mi organizzo?"
# 3. "Ho preso 28, 30 e 25 nei primi 3 esami, qual e' la media?"
# 4. "Ricordi quanti giorni mancano al mio esame?"  ← test memoria

print("=== Tutor universitario (scrivi 'esci' per uscire) ===\n")

while True:
    user_input = input("Tu: ")
    if user_input.strip().lower() in ("esci", "quit", "exit", ""):
        print("Sessione terminata. In bocca al lupo!")
        break
    try:
        response = tutor.run(user_input)
        print(f"Tutor: {response.content}\n")
    except Exception as e:
        print(f"Errore: {e}\n")


---
## Soluzione Esercizio 3 — Artigianale vs Framework: confronto ragionato

In [ ]:
# === Approccio 1: loop ReAct artigianale ===

from openai import OpenAI
from google.colab import userdata
from datetime import datetime
import json

client_oai = OpenAI(
    base_url="https://api.mistral.ai/v1",
    api_key=userdata.get("MISTRAL_API_KEY"),
)

def search_wikipedia(query: str) -> str:
    """Cerca informazioni su Wikipedia. Restituisce un breve estratto."""
    knowledge = {
        "leonardo da vinci": "Leonardo di ser Piero da Vinci (1452-1519) e' stato un inventore, artista e scienziato italiano del Rinascimento. Nato a Vinci, in Toscana, il 15 aprile 1452.",
        "albert einstein": "Albert Einstein (1879-1955) e' stato un fisico teorico tedesco. Nato a Ulm il 14 marzo 1879. Premio Nobel per la fisica nel 1921.",
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    return f"Nessun risultato trovato per: {query}"

def calculate(expression: str) -> str:
    """Calcola un'espressione matematica."""
    allowed = set("0123456789+-*/.(). ")
    if not all(c in allowed for c in expression):
        return "Errore: espressione non valida"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Errore: {e}"

def get_current_date() -> str:
    """Restituisce la data corrente."""
    now = datetime.now()
    return f"Oggi e' il {now.day}/{now.month}/{now.year}"

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "search_wikipedia",
            "description": "Cerca informazioni su Wikipedia. Restituisce un breve estratto.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Termine di ricerca"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Calcola un'espressione matematica.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Espressione matematica"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Restituisce la data corrente.",
            "parameters": {"type": "object", "properties": {}}
        }
    }
]

tool_registry = {
    "search_wikipedia": search_wikipedia,
    "calculate": calculate,
    "get_current_date": get_current_date,
}

def react_agent(question, tools, tool_registry, max_steps=10):
    messages = [
        {"role": "system", "content": "Sei un assistente. Usa i tool per rispondere. Cerca sempre le informazioni con search_wikipedia prima di rispondere."},
        {"role": "user", "content": question}
    ]
    for step in range(max_steps):
        response = client_oai.chat.completions.create(
            model="mistral-small-latest", messages=messages, tools=tools
        )
        msg = response.choices[0].message
        if response.choices[0].finish_reason == "stop":
            return msg.content
        if response.choices[0].finish_reason == "tool_calls":
            messages.append(msg)
            for tc in msg.tool_calls:
                fn = tool_registry[tc.function.name]
                args = json.loads(tc.function.arguments)
                result = fn(**args) if args else fn()
                print(f"  [ReAct] {tc.function.name}({args}) -> {result}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
    return "Timeout: troppi step"

DOMANDA = "In che anno e' nato Leonardo da Vinci e quanti anni fa e' stato?"

print("=== APPROCCIO 1: ReAct artigianale ===")
risposta_react = react_agent(DOMANDA, tools_schema, tool_registry)
print(f"Risposta: {risposta_react}")


In [ ]:
# === Approccio 2: stesso task con Agno ===

from agno.agent import Agent
from agno.models.mistral import MistralChat

# I tool search_wikipedia(), calculate() e get_current_date() sono gia' definiti sopra

agent_agno = Agent(
    model=MistralChat(id="mistral-small-latest"),
    tools=[search_wikipedia, calculate, get_current_date],
    instructions=["Sei un assistente. Usa i tool per rispondere. Cerca sempre le informazioni con search_wikipedia prima di rispondere."],
    markdown=True,
)

print("\n=== APPROCCIO 2: Agno ===")
agent_agno.print_response(DOMANDA, stream=True)


### Confronto (esempio di risposta attesa)

**Risultato**: entrambi gli approcci dovrebbero arrivare alla stessa risposta (Leonardo da Vinci nato nel 1452, circa 574 anni fa — valore riferito al 2026). La sequenza *ideale* è `search_wikipedia` → `get_current_date` → `calculate`, ma il modello può talvolta saltare uno step se ritiene di conoscere già una parte della risposta.

**Differenze chiave**:

| | ReAct artigianale | Agno |
|---|---|---|
| **Righe di codice** | ~60 (loop, schema JSON, registry, parsing) | ~8 (Agent + tools + print_response) |
| **JSON Schema** | scritto a mano per ogni tool | generato automaticamente dai type hints |
| **Loop** | implementato esplicitamente (while + if/else) | gestito internamente dal framework |
| **Tool call** | idealmente search → date → calculate | idealmente stessa sequenza (il modello può saltare step) |
| **Flessibilità** | totale — controllo su ogni step | limitata a ciò che il framework espone |
| **Debugging** | facile — ogni step è visibile nel codice | richiede `debug_mode=True` o logging del framework |

**Conclusione**: il framework riduce drasticamente il boilerplate, ma capire il loop artigianale (notebook 7A/7C) è fondamentale per sapere cosa succede "sotto il cofano" quando qualcosa non funziona.

---
## Soluzione Esercizio 4 — Tech News Analyst


In [ ]:
from agno.agent import Agent
from agno.models.mistral import MistralChat
from agno.tools.hackernews import HackerNewsTools
from agno.tools.wikipedia import WikipediaTools
import re


def classify_topic(title: str) -> str:
    """Classifica il titolo di una notizia tech in una categoria.

    Categorie possibili: AI, Web, Security, Hardware, Business, Other.
    Usa word boundary (\b) per evitare falsi positivi come 'ai' dentro 'mail'
    o 'arm' dentro 'farm'.
    """
    title_lower = title.lower()

    categories = {
        "AI": ["ai", "llm", "gpt", "model", "neural", "transformer", "machine learning",
               "deep learning", "openai", "anthropic", "gemini", "diffusion"],
        "Web": ["web", "browser", "css", "html", "javascript", "react", "frontend",
                "http", "api", "rest", "graphql"],
        "Security": ["security", "hack", "vulnerability", "cve", "encrypt",
                     "ransomware", "malware", "zero-day", "breach"],
        "Hardware": ["chip", "cpu", "gpu", "hardware", "risc", "arm", "intel",
                     "amd", "nvidia", "semiconductor", "quantum"],
        "Business": ["startup", "funding", "acquisition", "ipo", "revenue",
                     "valuation", "series a", "series b", "layoff"],
    }

    for category, keywords in categories.items():
        # word boundary per match su parole intere (evita 'ai' in 'mail')
        pattern = r"\b(" + "|".join(re.escape(kw) for kw in keywords) + r")\b"
        if re.search(pattern, title_lower):
            return category

    return "Other"


analyst = Agent(
    model=MistralChat(id="mistral-small-latest"),
    tools=[HackerNewsTools(), WikipediaTools(), classify_topic],
    instructions=[
        "Sei un analista tech. Scrivi sempre in italiano.",
        "Usa get_top_hackernews_stories per recuperare le notizie.",
        "Usa classify_topic per classificare ogni titolo in una categoria.",
        "Usa search_wikipedia per approfondire un argomento quando richiesto.",
        "Presenta i risultati in modo chiaro e strutturato.",
    ],
    markdown=True,
)

analyst.print_response(
    "Recupera le top 3 story da Hacker News, classifica ciascuna per categoria, "
    "poi cerca su Wikipedia il tema della prima story e scrivi un bollettino tech in italiano.",
    stream=True,
)


---
## Soluzione Esercizio 3bis — Agente ibrido web search + custom tool

In [ ]:
from agno.agent import Agent
from agno.models.mistral import MistralChat
from agno.tools.websearch import WebSearchTools

def calculate(expression: str) -> str:
    """Calcola un'espressione matematica. Supporta +, -, *, / e parentesi."""
    allowed = set("0123456789+-*/.(). ")
    if not all(c in allowed for c in expression):
        return "Errore: espressione non valida"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Errore: {e}"

# Un unico agente con web search + calcolatrice
agente_ibrido = Agent(
    model=MistralChat(id="mistral-small-latest"),
    tools=[WebSearchTools(), calculate],
    instructions=[
        "Sei un assistente che combina ricerca web e calcoli.",
        "Usa web_search per trovare dati reali (popolazioni, distanze, prezzi, ecc.)",
        "Usa calculate per fare elaborazioni numeriche sui dati trovati.",
        "Mostra sempre i passaggi: prima i dati trovati, poi il calcolo, poi il risultato.",
    ],
    markdown=True,
)

# Test: richiede sia ricerca che calcolo
agente_ibrido.print_response(
    "Cerca la distanza in km tra Roma e Milano, poi calcola quanto tempo ci vuole a 130 km/h",
    stream=True
)
